# Task 1 — EDA & Preprocessing: Fraud_Data.csv
**Adey Innovations Inc. — df Detection Project**

This notebook covers:
1. Data loading & inspection
2. Data cleaning
3. Exploratory Data Analysis (EDA)
4. IP → Country geolocation merge
5. Feature engineering
6. Encoding & scaling
7. Class imbalance handling (SMOTE)
8. Save processed dataset

In [4]:
import sys
!{sys.executable} -m pip list | findstr -i "scikit imbalanced"

imbalanced-learn          0.14.1
scikit-learn              1.6.1


In [5]:
import sys
print(sys.executable)

c:\Users\hp\Documents\10x\fraud-detection\.venv\Scripts\python.exe


In [6]:
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from preprocess import (
    load_fraud_data, load_ip_map,
    clean_fraud_data, merge_geolocation,
    engineer_features, encode_and_scale_fraud,
    split_and_resample
)
from eda_utils import (
    plot_class_distribution, plot_numeric_distributions,
    plot_categorical_fraud_rate, plot_top_fraud_countries,
    plot_time_since_signup, plot_hour_of_day,
    plot_velocity, plot_smote_comparison
)

sns.set_theme(style='whitegrid')
%matplotlib inline

### 1. Load and Inspect

In [7]:
df = load_fraud_data('../data/raw/fraud_data.csv')
ip_map = load_ip_map('../data/raw/IpAddress_to_country.csv')

print('Shape:', df.shape)
df.head()

INFO: Fraud_Data loaded: (151112, 11)
INFO: IP map loaded: (138846, 3)


Shape: (151112, 11)


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 151112 entries, 0 to 151111
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   user_id         151112 non-null  int64  
 1   signup_time     151112 non-null  str    
 2   purchase_time   151112 non-null  str    
 3   purchase_value  151112 non-null  int64  
 4   device_id       151112 non-null  str    
 5   source          151112 non-null  str    
 6   browser         151112 non-null  str    
 7   sex             151112 non-null  str    
 8   age             151112 non-null  int64  
 9   ip_address      151112 non-null  float64
 10  class           151112 non-null  int64  
dtypes: float64(1), int64(4), str(6)
memory usage: 12.7 MB


In [9]:
df.describe()

,user_id,purchase_value,age,ip_address,class
count,151112.000000,151112.000000,151112.000000,1.511120e+05,151112.000000
mean,200171.040970,36.935372,33.140704,2.152145e+09,0.093646
std,115369.285024,18.322762,8.617733,1.248497e+09,0.291336
min,2.000000,9.000000,18.000000,5.209350e+04,0.000000
25%,100642.500000,22.000000,27.000000,1.085934e+09,0.000000
50%,199958.000000,35.000000,33.000000,2.154770e+09,0.000000
75%,300054.000000,49.000000,39.000000,3.243258e+09,0.000000
max,400000.000000,154.000000,76.000000,4.294850e+09,1.000000


### 2. Data Cleaning


In [10]:
# Check nulls before cleaning
print('Null counts before cleaning:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')

Null counts before cleaning:
user_id           0
signup_time       0
purchase_time     0
purchase_value    0
device_id         0
source            0
browser           0
sex               0
age               0
ip_address        0
class             0
dtype: int64

Duplicate rows: 0


In [11]:
fraud_clean = clean_fraud_data(df)
print('Shape after cleaning:', fraud_clean.shape)
print('\nNull counts after cleaning:')
print(fraud_clean.isnull().sum())

INFO: Fraud_Data: dropped 0 duplicate rows
INFO: Null counts:
Series([], dtype: int64)
INFO: Fraud_Data after cleaning: (151112, 11)


Shape after cleaning: (151112, 11)

Null counts after cleaning:
user_id           0
signup_time       0
purchase_time     0
purchase_value    0
device_id         0
source            0
browser           0
sex               0
age               0
ip_address        0
class             0
dtype: int64
